In [5]:
import pandas as pd
import numpy as np
from os import path, listdir, makedirs
import scipy as sc
import matplotlib.pyplot as plt
from pyteomics import fasta
import triqler

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

# fasta dicts

In [6]:
fasta_path = './fasta/sprot_ecoli_ups_15072026_shuffled.fasta'

i = 0
double_ups_ecoli_organism_dict = {'Other':set()}
with open(fasta_path, mode='r') as f:
    for descr, seq in fasta.read(f) :
        if descr.startswith('sp|') :
            prot_id = descr.split('|')[1]
        else :
            prot_id = descr
        if 'HUMAN' in descr :
            double_ups_ecoli_organism_dict[prot_id] = 'Human'
            double_ups_ecoli_organism_dict[prot_id.split(' ')[0]] = 'Human'
        elif 'ECOLI' in descr :
            double_ups_ecoli_organism_dict[prot_id] = 'Ecoli'
        else :
            double_ups_ecoli_organism_dict['Other'].add(prot_id)

In [7]:
fasta_path = './fasta/human_ecoli_yeast_13072026_shuffled.fasta'

i = 0
double_lfqbench_organism_dict = {'Other':set()}
with open(fasta_path, mode='r') as f:
    for descr, seq in fasta.read(f) :
        if descr.startswith('sp|') :
            prot_id = descr.split('|')[1]
        else :
            prot_id = descr
        if 'HUMAN' in descr :
            double_lfqbench_organism_dict[prot_id] = 'Human'
        elif 'ECOLI' in descr :
            double_lfqbench_organism_dict[prot_id] = 'Ecoli'
        elif 'YEAST' in descr :
            double_lfqbench_organism_dict[prot_id] = 'Yeast'
        else :
            double_lfqbench_organism_dict['Other'].add(prot_id)

# tmp

In [13]:
organism_dict = double_ups_ecoli_organism_dict
colstarter = 'RD139_Wide_UPS1_'
colsplitter = '_inj'
search_dr = 'diann231_25fmol_vs_5fmol_rerun'
min_samples = 2
comp = '25fmol_vs_5fmol'

if search_dr.startswith('fragpipe') :
    inpath = path.join('./search_results', search_dr, comp, 'dia-quant-output', 'report.tsv')
    df = pd.read_csv(inpath, sep='\t', usecols=['Run', 'Protein.Ids', 'Protein.Group'])
else :
    inpath = path.join('./search_results', search_dr, comp, 'report.parquet')
    df = pd.read_parquet(inpath)
print(set(df['Run'].values))
all_columns = sorted(list(set(df['Run'].values)), 
                     key=lambda x: float(x.split(colstarter)[-1].split(colsplitter)[0].replace('fmol', '').replace('_', '.')) + 0.0001*int(x.split(colsplitter)[-1][0]),
                     reverse=False)

sample_path = inpath + '.sample_file.tsv'
conc_1, conc_2 = comp.replace('fmol', '').split('_vs_')
print(conc_1, conc_2)
fc = np.abs(0.5*min(np.log2(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), np.log2(10)))
samples = [path.basename(col) for col in all_columns if col.startswith(colstarter)]
temp_conds = [condstarter+col.split(colstarter)[-1].split(colsplitter)[0] for col in all_columns if col.startswith(colstarter)]
a = pd.DataFrame([samples, temp_conds], index=['run', 'condition']).T
a.to_csv(sample_path, sep='\t', index=False)
out_path = inpath + '.triqler_input.tsv'
print(sample_path, out_path, inpath, sep='\n')
!~/.pyenv/versions/3.10.11/envs/triqler31011/bin/python -m triqler.convert.diann --file_list_file $sample_path --out_file $out_path $inpath
infile = out_path
outfile = infile.replace('triqler_input', 'triqler_output_opt_mins3')
pref = 'DECOY_'
print(outfile)
if not 'diann' in search_dr :
    output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
    # fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
    # output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
else :
    output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
    # fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
    # output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile

{'RD139_Wide_UPS1_25fmol_inj1', 'RD139_Wide_UPS1_5fmol_inj2', 'RD139_Wide_UPS1_5fmol_inj1', 'RD139_Wide_UPS1_5fmol_inj3', 'RD139_Wide_UPS1_25fmol_inj3', 'RD139_Wide_UPS1_25fmol_inj2'}
25 5
./search_results/diann231_25fmol_vs_5fmol_rerun/25fmol_vs_5fmol/report.parquet.sample_file.tsv
./search_results/diann231_25fmol_vs_5fmol_rerun/25fmol_vs_5fmol/report.parquet.triqler_input.tsv
./search_results/diann231_25fmol_vs_5fmol_rerun/25fmol_vs_5fmol/report.parquet
triqler.convert.diann version 0.9.1
Copyright (c) 2018-2024 Matthew The, Patrick Truong. All rights reserved.
Written by:
- Matthew The (matthew.the@scilifelab.se)
- Patrick Truong (patrick.truong@scilifelab.se)
in the School of Engineering Sciences in Chemistry, Biotechnology and Health
at the Royal Institute of Technology in Stockholm.
Issued command: diann.py --file_list_file ./search_results/diann231_25fmol_vs_5fmol_rerun/25fmol_vs_5fmol/report.parquet.sample_file.tsv --out_file ./search_results/diann231_25fmol_vs_5fmol_rerun/25fm

In [32]:
!~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --help

Triqler version 0.9.1
Copyright (c) 2018-2024 Matthew The, Patrick Truong. All rights reserved.
Written by:
- Matthew The (matthew.the@scilifelab.se)
- Patrick Truong (patrick.truong@scilifelab.se)
in the School of Engineering Sciences in Chemistry, Biotechnology and Health
at the Royal Institute of Technology in Stockholm.
Issued command: triqler.py --help
usage: triqler [-h] [--out_file OUT] [--fold_change_eval F]
               [--decoy_pattern P] [--protein_name_separator P]
               [--missing_value_prior D] [--min_samples N] [--num_threads N]
               [--ttest] [--write_spectrum_quants]
               [--write_protein_posteriors P_OUT]
               [--write_group_posteriors G_OUT]
               [--write_fold_change_posteriors F_OUT]
               [--csv-field-size-limit CSV_FIELD_SIZE_LIMIT]
               IN_FILE

positional arguments:
  IN_FILE               List of PSMs with abundances (not log transformed!)
                        and search engine score. See 

In [14]:
organism_dict = double_ups_ecoli_organism_dict
search_engine = 'diann231'
outfile = './search_results/diann231_25fmol_vs_5fmol_rerun/25fmol_vs_5fmol/report.parquet.triqler_output_opt_mins3.tsv'

temp=pd.read_csv(outfile, sep='^', header=None,)
l = len(temp[0][0].split('\t'))
# print(l)
temp2=pd.DataFrame(list(temp[0].apply(lambda x: x.split('\t', maxsplit=l-1, ))),)
temp2.columns = temp2.loc[0, :]
outdf = temp2.loc[1:, :].copy()
fc = [float(x.split('diff_exp_prob_')[-1]) for x in outdf.columns if x.startswith('diff_exp_prob_')][0]
excl = ['protein', 'peptides', ]
for col in outdf.columns :
    if not col in excl : 
        outdf[col] = outdf[col].astype(float)
outdf['peptides'] = outdf['peptides'].str.replace('\t', ';')
outdf['FDR_pass'] = outdf['q_value'] < 0.05
outdf['FC_pass'] = abs(outdf['log2_fold_change']) >= fc
outdf['Organism'] = outdf['protein'].map(organism_dict)
outdf = outdf[~outdf['protein'].str.startswith('DECOY')].copy()
outdf = outdf.rename(columns={ 'log2_fold_change':'log2_FC', 'protein':'Protein.Group'})
outdf['Human'] = outdf['Organism'] == 'Human'
outdf['Ecoli'] = outdf['Organism'] == 'Ecoli'
outdf['Yeast'] = outdf['Organism'] == 'Yeast'
outdf['Search Engine'] = [search_engine for _ in range(len(outdf))]
outdf['Quantitation'] = ['Triqler' for _ in range(len(outdf))]
outdf['Imputation'] = ['Triqler' for _ in range(len(outdf))]
outdf['FC_thr'] = [fc for _ in range(len(outdf))]
for org in ['Human', 'Ecoli', 'Yeast'] :
    print(org, len(outdf[(outdf['FDR_pass']) & (outdf['FC_pass']) & (outdf[org])]) )
outdf.head()

Human 36
Ecoli 0
Yeast 0


,q_value,posterior_error_prob,Protein.Group,num_peptides,protein_id_posterior_error_prob,log2_FC,diff_exp_prob_1.160964047443681,Condition_25fmol:RD139_Wide_UPS1_25fmol_inj1,Condition_25fmol:RD139_Wide_UPS1_25fmol_inj2,Condition_25fmol:RD139_Wide_UPS1_25fmol_inj3,Condition_5fmol:RD139_Wide_UPS1_5fmol_inj1,Condition_5fmol:RD139_Wide_UPS1_5fmol_inj2,Condition_5fmol:RD139_Wide_UPS1_5fmol_inj3,peptides,FDR_pass,FC_pass,Organism,Human,Ecoli,Yeast,Search Engine,Quantitation,Imputation,FC_thr
1,1.518000e-08,1.518000e-08,P02788ups|TRFL_HUMAN_UPS,22.0,8.734000e-09,2.355,6.442000e-09,2.319,2.138,2.335,0.4418,0.4459,0.4386,IDSGLYLGSGYFTAIQNLR;LRPVAAEVYGTER;ADAVTLDGGFIY...,True,True,Human,True,False,False,diann231,Triqler,Triqler,1.160964
2,1.992000e-08,2.466000e-08,P12081ups|SYHC_HUMAN_UPS,20.0,1.434000e-08,2.345,1.033000e-08,2.261,2.120,2.389,0.4356,0.4608,0.4352,HGAEVIDTPVFELK;YDLTVPFAR;AALEELVK;ASAELIEEEVAK...,True,True,Human,True,False,False,diann231,Triqler,Triqler,1.160964
3,2.821000e-08,4.480000e-08,P06732ups|KCRM_HUMAN_UPS,16.0,3.263000e-08,2.299,1.217000e-08,2.232,2.155,2.266,0.4286,0.4763,0.4495,GTGGVDTAAVGSVFDVSNADR;LGSSEVEQVQLVVDGVK;LSVEAL...,True,True,Human,True,False,False,diann231,Triqler,Triqler,1.160964
4,3.659000e-08,6.172000e-08,P08758ups|ANXA5_HUMAN_UPS,16.0,4.975000e-08,2.340,1.197000e-08,2.275,2.105,2.364,0.4419,0.4636,0.4310,GLGTDEESILTLLTSR;GAGTDDHTLIR;GTVTDFPGFDER;QVYE...,True,True,Human,True,False,False,diann231,Triqler,Triqler,1.160964
5,4.168000e-08,6.204000e-08,P02768ups|ALBU_HUMAN_UPS,20.0,2.811000e-08,2.296,3.393000e-08,2.262,2.031,2.365,0.4401,0.4740,0.4411,KVPQVSTPTLVEVSR;RHPDYSVVLLLR;RHPYFYAPELLFFAK;A...,True,True,Human,True,False,False,diann231,Triqler,Triqler,1.160964


# UPS-Ecoli

## UPS-Ecoli standart fc threshold

In [18]:
organism_dict = double_ups_ecoli_organism_dict
python_path = ''
colstarter = 'RD139_Wide_UPS1_'
colsplitter = '_inj'
condstarter = ''
min_samples = 3
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'ups_ecoli' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and 'diann' in search_dr :
    if 'ups_ecoli' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or 'diann231' in search_dr) :
        search_engine = search_dr.replace('ups_ecoli', '').replace('fragpipe24', '').replace('triqler', '').replace('_', '')
        for comp in listdir(path.join('./search_results', search_dr)) :
            if '_vs_' in comp : 
                if search_dr.startswith('fragpipe') :
                    inpath = path.join('./search_results', search_dr, comp, 'dia-quant-output', 'report.tsv')
                    df = pd.read_csv(inpath, sep='\t', usecols=['Run', 'Protein.Ids', 'Protein.Group'])
                else :
                    inpath = path.join('./search_results', search_dr, comp, 'report.parquet')
                    df = pd.read_parquet(inpath)
                print(set(df['Run'].values))
                all_columns = sorted(list(set(df['Run'].values)), 
                                     key=lambda x: float(x.split(colstarter)[-1].split(colsplitter)[0].replace('fmol', '').replace('_', '.')) + 0.0001*int(x.split(colsplitter)[-1][0]),
                                     reverse=False)

                sample_path = inpath + '.sample_file.tsv'
                conc_1, conc_2 = comp.replace('fmol', '').split('_vs_')
                print(conc_1, conc_2)
                fc = np.abs(0.5*min(np.log2(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), np.log2(10)))
                samples = [path.basename(col) for col in all_columns if col.startswith(colstarter)]
                temp_conds = [condstarter+col.split(colstarter)[-1].split(colsplitter)[0] for col in all_columns if col.startswith(colstarter)]
                a = pd.DataFrame([samples, temp_conds], index=['run', 'condition']).T
                a.to_csv(sample_path, sep='\t', index=False)
                out_path = inpath + '.triqler_input.tsv'
                print(sample_path, out_path, inpath, sep='\n')
                !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/python -m triqler.convert.diann --file_list_file $sample_path --out_file $out_path $inpath
                infile = out_path
                outfile = infile.replace('triqler_input', 'triqler_output')
                pref = 'DECOY_'
                print(outfile)
                if not 'diann' in search_dr :
                    output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
                    # fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
                    # output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples 2 --fold_change_eval $fc $infile
                else :
                    output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
                    # fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
                    # output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples 2 --fold_change_eval $fc $infile

                

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

fragpipe24_LFQbenchTOF_diatracer_triqler 

diann181_ups_ecoli_pg_intensities 

tmp 

fragpipe24_LFQbench_umpire 

.ipynb_checkpoints 

fragpipe24_ups_ecoli_msfragger_triqler 

{'RD139_Wide_UPS1_5fmol_inj1', 'RD139_Wide_UPS1_0_1fmol_inj3', 'RD139_Wide_UPS1_5fmol_inj2', 'RD139_Wide_UPS1_0_1fmol_inj2', 'RD139_Wide_UPS1_5fmol_inj3', 'RD139_Wide_UPS1_0_1fmol_inj1'}
5 0_1
./search_results/fragpipe24_ups_ecoli_msfragger_triqler/5fmol_vs_0_1fmol/dia-quant-output/report.tsv.sample_file.tsv
./search_results/fragpipe24_ups_ecoli_msfragger_triqler/5fmol_vs_0_1fmol/dia-quant-output/report.tsv.triqler_input.tsv
./search_results/fragpipe24_ups_ecoli_msfragger_triqler/5fmol_vs_0_1fmol/dia-quant-output/report.tsv
triqler.convert.diann version 0.9.1
Copyright (c) 2018-2024 Matthew The, Patrick Truong. All rights reserved.
Written by:
- Matthew Th

In [19]:
organism_dict = double_ups_ecoli_organism_dict
python_path = ''
colstarter = 'RD139_Wide_UPS1_'
colsplitter = '_inj'
condstarter = ''
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'ups_ecoli' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and 'diann' in search_dr :
    if 'ups_ecoli' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or 'diann231' in search_dr) :
        search_engine = search_dr.replace('ups_ecoli', '').replace('fragpipe24', '').replace('triqler', '').replace('_', '')
        for comp in listdir(path.join('./search_results', search_dr)) :
            if '_vs_' in comp : 
                if search_dr.startswith('fragpipe') :
                    inpath = path.join('./search_results', search_dr, comp, 'dia-quant-output', 'report.tsv')
                    df = pd.read_csv(inpath, sep='\t', usecols=['Run', 'Protein.Ids', 'Protein.Group'])
                else :
                    inpath = path.join('./search_results', search_dr, comp, 'report.parquet')
                    df = pd.read_parquet(inpath)
                conc_1, conc_2 = comp.replace('fmol', '').split('_vs_')
                # fc = np.abs(0.5*min(np.log2(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), np.log2(10)))
                outfile = inpath + '.triqler_output.tsv'
                # outdf = pd.read_csv(outfile, sep='\t')
                temp=pd.read_csv(outfile, sep='^', header=None,)
                l = len(temp[0][0].split('\t'))
                # print(l)
                temp2=pd.DataFrame(list(temp[0].apply(lambda x: x.split('\t', maxsplit=l-1, ))),)
                temp2.columns = temp2.loc[0, :]
                outdf = temp2.loc[1:, :].copy()
                fc = [float(x.split('diff_exp_prob_')[-1]) for x in outdf.columns if x.startswith('diff_exp_prob_')][0]
                excl = ['protein', 'peptides', ]
                for col in outdf.columns :
                    if not col in excl : 
                        outdf[col] = outdf[col].astype(float)
                outdf['peptides'] = outdf['peptides'].str.replace('\t', ';')
                
                outdf['FDR_pass'] = outdf['q_value'] < 0.05
                outdf['FC_pass'] = np.abs(outdf['log2_fold_change']) >= fc
                outdf['Organism'] = outdf['protein'].map(organism_dict)
                outdf = outdf[~outdf['protein'].str.startswith('DECOY')].copy()
                outdf = outdf.rename(columns={ 'log2_fold_change':'log2_FC', 'protein':'Protein.Group'})
                outdf['Human'] = outdf['Organism'] == 'Human'
                outdf['Ecoli'] = outdf['Organism'] == 'Ecoli'
                outdf['Yeast'] = outdf['Organism'] == 'Yeast'
                outdf['Search Engine'] = [search_engine for _ in range(len(outdf))]
                outdf['Quantitation'] = ['Triqler' for _ in range(len(outdf))]
                outdf['Imputation'] = ['Triqler' for _ in range(len(outdf))]
                outdf['FC_thr'] = [fc for _ in range(len(outdf))]
                savepath = './quantitation/'+search_dr.replace('_triqler', '')+'/'+comp+'_triqler.tsv'
                # print(savepath)
                makedirs(path.dirname(savepath), exist_ok=True)
                outdf.to_csv(savepath, index=False, sep='\t')

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

fragpipe24_LFQbenchTOF_diatracer_triqler 

diann181_ups_ecoli_pg_intensities 

tmp 

fragpipe24_LFQbench_umpire 

.ipynb_checkpoints 

fragpipe24_ups_ecoli_msfragger_triqler 

diann231_ups_ecoli_triqler 

fragpipe24_LFQbench_umpire_triqler 

diann231_25fmol_vs_5fmol_rerun 

fragpipe24_LFQbench.sh 

diann231_LFQbench 

fragpipe24_ups_ecoli_umpire_triqler 

fragpipe24_ups_ecoli_msfragger 

diann231_ups_ecoli 

fragpipe24_UPS-Ecoli.sh 



## UPS-Ecoli standart fc threshold mins 2

In [25]:
organism_dict = double_ups_ecoli_organism_dict
python_path = ''
colstarter = 'RD139_Wide_UPS1_'
colsplitter = '_inj'
condstarter = ''
min_samples = 2
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'ups_ecoli' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and 'diann' in search_dr :
    if search_dr == 'diann231_ups_ecoli' :
        search_engine = search_dr.replace('ups_ecoli', '').replace('fragpipe24', '').replace('triqler', '').replace('_', '')
        for comp in listdir(path.join('./search_results', search_dr)) :
            if '_vs_' in comp : 
                if search_dr.startswith('fragpipe') :
                    inpath = path.join('./search_results', search_dr, comp, 'dia-quant-output', 'report.tsv')
                    df = pd.read_csv(inpath, sep='\t', usecols=['Run', 'Protein.Ids', 'Protein.Group'])
                else :
                    inpath = path.join('./search_results', search_dr, comp, 'report.parquet')
                    df = pd.read_parquet(inpath)
                print(set(df['Run'].values))
                all_columns = sorted(list(set(df['Run'].values)), 
                                     key=lambda x: float(x.split(colstarter)[-1].split(colsplitter)[0].replace('fmol', '').replace('_', '.')) + 0.0001*int(x.split(colsplitter)[-1][0]),
                                     reverse=False)

                sample_path = inpath + '.sample_file.tsv'
                conc_1, conc_2 = comp.replace('fmol', '').split('_vs_')
                print(conc_1, conc_2)
                fc = np.abs(0.5*min(np.log2(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), np.log2(10)))
                samples = [path.basename(col) for col in all_columns if col.startswith(colstarter)]
                temp_conds = [condstarter+col.split(colstarter)[-1].split(colsplitter)[0] for col in all_columns if col.startswith(colstarter)]
                a = pd.DataFrame([samples, temp_conds], index=['run', 'condition']).T
                a.to_csv(sample_path, sep='\t', index=False)
                out_path = inpath + '.triqler_input.tsv'
                print(sample_path, out_path, inpath, sep='\n')
                !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/python -m triqler.convert.diann --file_list_file $sample_path --out_file $out_path $inpath
                infile = out_path
                outfile = infile.replace('triqler_input', 'triqler_output_mins2')
                pref = 'DECOY_'
                print(outfile)
                if not 'diann' in search_dr :
                    output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
                    # fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
                    # output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples 2 --fold_change_eval $fc $infile
                else :
                    output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
                    # fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
                    # output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples 2 --fold_change_eval $fc $infile

                

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

fragpipe24_LFQbenchTOF_diatracer_triqler 

diann181_ups_ecoli_pg_intensities 

tmp 

fragpipe24_LFQbench_umpire 

.ipynb_checkpoints 

fragpipe24_ups_ecoli_msfragger_triqler 

diann231_ups_ecoli_triqler 

fragpipe24_LFQbench_umpire_triqler 

diann231_25fmol_vs_5fmol_rerun 

fragpipe24_LFQbench.sh 

diann231_LFQbench 

fragpipe24_ups_ecoli_umpire_triqler 

fragpipe24_ups_ecoli_msfragger 

diann231_ups_ecoli 

{'RD139_Wide_UPS1_0_25fmol_inj1', 'RD139_Wide_UPS1_50fmol_inj3', 'RD139_Wide_UPS1_0_25fmol_inj3', 'RD139_Wide_UPS1_0_25fmol_inj2', 'RD139_Wide_UPS1_50fmol_inj1', 'RD139_Wide_UPS1_50fmol_inj2'}
50 0_25
./search_results/diann231_ups_ecoli/50fmol_vs_0_25fmol/report.parquet.sample_file.tsv
./search_results/diann231_ups_ecoli/50fmol_vs_0_25fmol/report.parquet.triqler_input.tsv
./search_results/diann231_ups_ecoli/50fmol_vs_0_25fmo

In [26]:
organism_dict = double_ups_ecoli_organism_dict
python_path = ''
colstarter = 'RD139_Wide_UPS1_'
colsplitter = '_inj'
condstarter = ''
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'ups_ecoli' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and 'diann' in search_dr :
    if search_dr == 'diann231_ups_ecoli' :
        search_engine = search_dr.replace('ups_ecoli', '').replace('fragpipe24', '').replace('triqler', '').replace('_', '')
        for comp in listdir(path.join('./search_results', search_dr)) :
            if '_vs_' in comp : 
                if search_dr.startswith('fragpipe') :
                    inpath = path.join('./search_results', search_dr, comp, 'dia-quant-output', 'report.tsv')
                    df = pd.read_csv(inpath, sep='\t', usecols=['Run', 'Protein.Ids', 'Protein.Group'])
                else :
                    inpath = path.join('./search_results', search_dr, comp, 'report.parquet')
                    df = pd.read_parquet(inpath)
                conc_1, conc_2 = comp.replace('fmol', '').split('_vs_')
                # fc = np.abs(0.5*min(np.log2(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), np.log2(10)))
                outfile = inpath + '.triqler_output_mins2.tsv'
                # outdf = pd.read_csv(outfile, sep='\t')
                temp=pd.read_csv(outfile, sep='^', header=None,)
                l = len(temp[0][0].split('\t'))
                # print(l)
                temp2=pd.DataFrame(list(temp[0].apply(lambda x: x.split('\t', maxsplit=l-1, ))),)
                temp2.columns = temp2.loc[0, :]
                outdf = temp2.loc[1:, :].copy()
                fc = [float(x.split('diff_exp_prob_')[-1]) for x in outdf.columns if x.startswith('diff_exp_prob_')][0]
                excl = ['protein', 'peptides', ]
                for col in outdf.columns :
                    if not col in excl : 
                        outdf[col] = outdf[col].astype(float)
                outdf['peptides'] = outdf['peptides'].str.replace('\t', ';')
                
                outdf['FDR_pass'] = outdf['q_value'] < 0.05
                outdf['FC_pass'] = np.abs(outdf['log2_fold_change']) >= fc
                outdf['Organism'] = outdf['protein'].map(organism_dict)
                outdf = outdf[~outdf['protein'].str.startswith('DECOY')].copy()
                outdf = outdf.rename(columns={ 'log2_fold_change':'log2_FC', 'protein':'Protein.Group'})
                outdf['Human'] = outdf['Organism'] == 'Human'
                outdf['Ecoli'] = outdf['Organism'] == 'Ecoli'
                outdf['Yeast'] = outdf['Organism'] == 'Yeast'
                outdf['Search Engine'] = [search_engine for _ in range(len(outdf))]
                outdf['Quantitation'] = ['Triqler' for _ in range(len(outdf))]
                outdf['Imputation'] = ['Triqler' for _ in range(len(outdf))]
                outdf['FC_thr'] = [fc for _ in range(len(outdf))]
                savepath = './quantitation/'+search_dr.replace('_triqler', '')+'/'+comp+'_triqler_mins2.tsv'
                # print(savepath)
                makedirs(path.dirname(savepath), exist_ok=True)
                outdf.to_csv(savepath, index=False, sep='\t')

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

fragpipe24_LFQbenchTOF_diatracer_triqler 

diann181_ups_ecoli_pg_intensities 

tmp 

fragpipe24_LFQbench_umpire 

.ipynb_checkpoints 

fragpipe24_ups_ecoli_msfragger_triqler 

diann231_ups_ecoli_triqler 

fragpipe24_LFQbench_umpire_triqler 

diann231_25fmol_vs_5fmol_rerun 

fragpipe24_LFQbench.sh 

diann231_LFQbench 

fragpipe24_ups_ecoli_umpire_triqler 

fragpipe24_ups_ecoli_msfragger 

diann231_ups_ecoli 

fragpipe24_UPS-Ecoli.sh 



## UPS-Ecoli triqler opt fc threshold

In [159]:
organism_dict = double_ups_ecoli_organism_dict
python_path = ''
colstarter = 'RD139_Wide_UPS1_'
colsplitter = '_inj'
condstarter = ''
min_samples = 3
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'ups_ecoli' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and 'diann' in search_dr :
    if 'ups_ecoli' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or 'diann231' in search_dr) :
        search_engine = search_dr.replace('ups_ecoli', '').replace('fragpipe24', '').replace('triqler', '').replace('_', '')
        for comp in listdir(path.join('./search_results', search_dr)) :
            if '_vs_' in comp : 
                if search_dr.startswith('fragpipe') :
                    inpath = path.join('./search_results', search_dr, comp, 'dia-quant-output', 'report.tsv')
                    df = pd.read_csv(inpath, sep='\t', usecols=['Run', 'Protein.Ids', 'Protein.Group'])
                else :
                    inpath = path.join('./search_results', search_dr, comp, 'report.parquet')
                    df = pd.read_parquet(inpath)
                print(set(df['Run'].values))
                all_columns = sorted(list(set(df['Run'].values)), 
                                     key=lambda x: float(x.split(colstarter)[-1].split(colsplitter)[0].replace('fmol', '').replace('_', '.')) + 0.0001*int(x.split(colsplitter)[-1][0]),
                                     reverse=False)

                sample_path = inpath + '.sample_file.tsv'
                conc_1, conc_2 = comp.replace('fmol', '').split('_vs_')
                print(conc_1, conc_2)
                fc = np.abs(0.5*min(np.log2(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), np.log2(10)))
                samples = [path.basename(col) for col in all_columns if col.startswith(colstarter)]
                temp_conds = [condstarter+col.split(colstarter)[-1].split(colsplitter)[0] for col in all_columns if col.startswith(colstarter)]
                a = pd.DataFrame([samples, temp_conds], index=['run', 'condition']).T
                a.to_csv(sample_path, sep='\t', index=False)
                out_path = inpath + '.triqler_input.tsv'
                print(sample_path, out_path, inpath, sep='\n')
                !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/python -m triqler.convert.diann --file_list_file $sample_path --out_file $out_path $inpath
                infile = out_path
                outfile = infile.replace('triqler_input', 'triqler_output_opt')
                pref = 'DECOY_'
                print(outfile)
                if not 'diann' in search_dr :
                    output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
                    fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
                    output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
                else :
                    output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
                    fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
                    output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile

                

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

fragpipe24_LFQbenchTOF_diatracer_triqler 

diann181_ups_ecoli_pg_intensities 

tmp 

fragpipe24_LFQbench_umpire 

.ipynb_checkpoints 

fragpipe24_ups_ecoli_msfragger_triqler 

{'RD139_Wide_UPS1_5fmol_inj2', 'RD139_Wide_UPS1_5fmol_inj3', 'RD139_Wide_UPS1_5fmol_inj1', 'RD139_Wide_UPS1_0_1fmol_inj2', 'RD139_Wide_UPS1_0_1fmol_inj1', 'RD139_Wide_UPS1_0_1fmol_inj3'}
5 0_1
./search_results/fragpipe24_ups_ecoli_msfragger_triqler/5fmol_vs_0_1fmol/dia-quant-output/report.tsv.sample_file.tsv
./search_results/fragpipe24_ups_ecoli_msfragger_triqler/5fmol_vs_0_1fmol/dia-quant-output/report.tsv.triqler_input.tsv
./search_results/fragpipe24_ups_ecoli_msfragger_triqler/5fmol_vs_0_1fmol/dia-quant-output/report.tsv
triqler.convert.diann version 0.9.1
Copyright (c) 2018-2024 Matthew The, Patrick Truong. All rights reserved.
Written by:
- Matthew Th

In [160]:
organism_dict = double_ups_ecoli_organism_dict
python_path = ''
colstarter = 'RD139_Wide_UPS1_'
colsplitter = '_inj'
condstarter = ''
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'ups_ecoli' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and 'diann' in search_dr :
    if 'ups_ecoli' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or 'diann231' in search_dr) :
        search_engine = search_dr.replace('ups_ecoli', '').replace('fragpipe24', '').replace('triqler', '').replace('_', '')
        for comp in listdir(path.join('./search_results', search_dr)) :
            if '_vs_' in comp : 
                if search_dr.startswith('fragpipe') :
                    inpath = path.join('./search_results', search_dr, comp, 'dia-quant-output', 'report.tsv')
                    df = pd.read_csv(inpath, sep='\t', usecols=['Run', 'Protein.Ids', 'Protein.Group'])
                else :
                    inpath = path.join('./search_results', search_dr, comp, 'report.parquet')
                    df = pd.read_parquet(inpath)
                conc_1, conc_2 = comp.replace('fmol', '').split('_vs_')
                # fc = np.abs(0.5*min(np.log2(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), np.log2(10)))
                outfile = inpath + '.triqler_output_opt.tsv'
                # outdf = pd.read_csv(outfile, sep='\t')
                temp=pd.read_csv(outfile, sep='^', header=None,)
                l = len(temp[0][0].split('\t'))
                # print(l)
                temp2=pd.DataFrame(list(temp[0].apply(lambda x: x.split('\t', maxsplit=l-1, ))),)
                temp2.columns = temp2.loc[0, :]
                outdf = temp2.loc[1:, :].copy()
                fc = [float(x.split('diff_exp_prob_')[-1]) for x in outdf.columns if x.startswith('diff_exp_prob_')][0]
                excl = ['protein', 'peptides', ]
                for col in outdf.columns :
                    if not col in excl : 
                        outdf[col] = outdf[col].astype(float)
                outdf['peptides'] = outdf['peptides'].str.replace('\t', ';')
                
                outdf['FDR_pass'] = outdf['q_value'] < 0.05
                outdf['FC_pass'] = np.abs(outdf['log2_fold_change']) >= fc
                outdf['Organism'] = outdf['protein'].map(organism_dict)
                outdf = outdf[~outdf['protein'].str.startswith('DECOY')].copy()
                outdf = outdf.rename(columns={ 'log2_fold_change':'log2_FC', 'protein':'Protein.Group'})
                outdf['Human'] = outdf['Organism'] == 'Human'
                outdf['Ecoli'] = outdf['Organism'] == 'Ecoli'
                outdf['Yeast'] = outdf['Organism'] == 'Yeast'
                outdf['Search Engine'] = [search_engine for _ in range(len(outdf))]
                outdf['Quantitation'] = ['Triqler' for _ in range(len(outdf))]
                outdf['Imputation'] = ['Triqler' for _ in range(len(outdf))]
                outdf['FC_thr'] = [fc for _ in range(len(outdf))]
                savepath = './quantitation/'+search_dr.replace('_triqler', '')+'/'+comp+'_triqler_opt.tsv'
                # print(savepath)
                makedirs(path.dirname(savepath), exist_ok=True)
                outdf.to_csv(savepath, index=False, sep='\t')

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

fragpipe24_LFQbenchTOF_diatracer_triqler 

diann181_ups_ecoli_pg_intensities 

tmp 

fragpipe24_LFQbench_umpire 

.ipynb_checkpoints 

fragpipe24_ups_ecoli_msfragger_triqler 

diann231_ups_ecoli_triqler 

fragpipe24_LFQbench_umpire_triqler 

diann231_25fmol_vs_5fmol_rerun 

fragpipe24_LFQbench.sh 

diann231_LFQbench 

fragpipe24_ups_ecoli_umpire_triqler 

fragpipe24_ups_ecoli_msfragger 

diann231_ups_ecoli 

fragpipe24_UPS-Ecoli.sh 



## UPS-Ecoli double fasta diann231 copy to quantitation

In [27]:
organism_dict = double_ups_ecoli_organism_dict
python_path = ''
colstarter = 'RD139_Wide_UPS1_'
colsplitter = '_inj'
condstarter = ''
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'ups_ecoli' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and 'diann' in search_dr :
    if search_dr == 'diann231_ups_ecoli_triqler' :
        search_engine = search_dr.replace('ups_ecoli', '').replace('fragpipe24', '').replace('triqler', '').replace('_', '')
        for opt in ['_opt', ''] :
            for comp in listdir(path.join('./search_results', search_dr)) :
                if '_vs_' in comp : 
                    if search_dr.startswith('fragpipe') :
                        inpath = path.join('./search_results', search_dr, comp, 'dia-quant-output', 'report.tsv')
                        df = pd.read_csv(inpath, sep='\t', usecols=['Run', 'Protein.Ids', 'Protein.Group'])
                    else :
                        inpath = path.join('./search_results', search_dr, comp, 'report.parquet')
                        df = pd.read_parquet(inpath)
                    conc_1, conc_2 = comp.replace('fmol', '').split('_vs_')
                    # fc = np.abs(0.5*min(np.log2(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), np.log2(10)))
                    outfile = inpath + '.triqler_output{}.tsv'.format(opt)
                    # outdf = pd.read_csv(outfile, sep='\t')
                    temp=pd.read_csv(outfile, sep='^', header=None,)
                    l = len(temp[0][0].split('\t'))
                    # print(l)
                    temp2=pd.DataFrame(list(temp[0].apply(lambda x: x.split('\t', maxsplit=l-1, ))),)
                    temp2.columns = temp2.loc[0, :]
                    outdf = temp2.loc[1:, :].copy()
                    fc = [float(x.split('diff_exp_prob_')[-1]) for x in outdf.columns if x.startswith('diff_exp_prob_')][0]
                    excl = ['protein', 'peptides', ]
                    for col in outdf.columns :
                        if not col in excl : 
                            outdf[col] = outdf[col].astype(float)
                    outdf['peptides'] = outdf['peptides'].str.replace('\t', ';')
                    
                    outdf['FDR_pass'] = outdf['q_value'] < 0.05
                    outdf['FC_pass'] = np.abs(outdf['log2_fold_change']) >= fc
                    outdf['Organism'] = outdf['protein'].map(organism_dict)
                    outdf = outdf[~outdf['protein'].str.startswith('DECOY')].copy()
                    outdf = outdf.rename(columns={ 'log2_fold_change':'log2_FC', 'protein':'Protein.Group'})
                    outdf['Human'] = outdf['Organism'] == 'Human'
                    outdf['Ecoli'] = outdf['Organism'] == 'Ecoli'
                    outdf['Yeast'] = outdf['Organism'] == 'Yeast'
                    outdf['Search Engine'] = [search_engine for _ in range(len(outdf))]
                    outdf['Quantitation'] = ['Triqler' for _ in range(len(outdf))]
                    outdf['Imputation'] = ['Triqler' for _ in range(len(outdf))]
                    outdf['FC_thr'] = [fc for _ in range(len(outdf))]
                    savepath = './quantitation/'+search_dr.replace('_triqler', '')+'/'+comp+'_triqler{}_double_fasta.tsv'.format(opt)
                    # print(savepath)
                    makedirs(path.dirname(savepath), exist_ok=True)
                    outdf.to_csv(savepath, index=False, sep='\t')

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

fragpipe24_LFQbenchTOF_diatracer_triqler 

diann181_ups_ecoli_pg_intensities 

tmp 

fragpipe24_LFQbench_umpire 

.ipynb_checkpoints 

fragpipe24_ups_ecoli_msfragger_triqler 

diann231_ups_ecoli_triqler 

fragpipe24_LFQbench_umpire_triqler 

diann231_25fmol_vs_5fmol_rerun 

fragpipe24_LFQbench.sh 

diann231_LFQbench 

fragpipe24_ups_ecoli_umpire_triqler 

fragpipe24_ups_ecoli_msfragger 

diann231_ups_ecoli 

fragpipe24_UPS-Ecoli.sh 



# LFQbench Orbitrap

## LFQbench Orbitrap triqler standart fc threshold

In [20]:
min_samples = 9
python_path = ''
colstarter = 'LFQ_Orbitrap_AIF_Condition_'
colsplitter = '_Sample'
condstarter = 'Condition_'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'LFQbench' in search_dr and path.isdir(path.join('./search_results', search_dr)) and search_dr.startswith('diann') :
    if 'LFQbench' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or search_dr.startswith('diann')) :
        # for comp in listdir(path.join('./search_results', search_dr)) :
        #     if '_vs_' in comp : 
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
            df = pd.read_csv(inpath, sep='\t', usecols=['Run'])
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet')
            df = pd.read_parquet(inpath)
        print(set(df['Run'].values))
        all_columns = sorted(list(set(df['Run'].values)), 
                             key=lambda x: x.split(colstarter)[-1].split(colsplitter)[0].replace('_', '.') + x.split(colsplitter)[-1][0],
                             reverse=False)

        sample_path = inpath + '.sample_file.tsv'
        # conc_1, conc_2 = comp.replace('fmol', '').split('_vs_')
        # print(conc_1, conc_2)
        # fc = np.abs(0.5*min(np.log2(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), np.log2(10)))
        fc = 0.5
        samples = sorted([path.basename(col) for col in all_columns if col.startswith(colstarter)], 
                         reverse=False)
        temp_conds = sorted([condstarter+col.split(colstarter)[-1].split(colsplitter)[0] for col in all_columns if col.startswith(colstarter)],
                            reverse=False)
        a = pd.DataFrame([samples, temp_conds], index=['run', 'condition']).T
        # print(a)
        a.to_csv(sample_path, sep='\t', index=False)
        out_path = inpath + '.triqler_input.tsv'
        print(sample_path, out_path, inpath, sep='\n')
        !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/python -m triqler.convert.diann --file_list_file $sample_path --out_file $out_path $inpath
        infile = out_path
        outfile = infile.replace('triqler_input', 'triqler_output')
        pref = 'DECOY_'
        print(outfile)
        if not 'diann' in search_dr :
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
            # fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
            # output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
        else :
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
            # fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
            # output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

{'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_02', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_03', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_01', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_02', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_03', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_01', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_01', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Beta_02', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_01', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Gamma_01', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_03', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_02', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Beta_03', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Gamma_03', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_03', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_02', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Gamma_02', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Beta_01'}
./search_results/fragpipe24_LFQbench_msfragger_triqler/dia-quant-output/report

In [21]:
organism_dict = double_lfqbench_organism_dict
colstarter = 'LFQ_Orbitrap_AIF_Condition_'
colsplitter = '_Sample'
condstarter = 'Condition_'
fc = 0.5
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'LFQbench' in search_dr and path.isdir(path.join('./search_results', search_dr)) and search_dr.startswith('diann') :
    if 'LFQbench' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or 'diann' in search_dr) :
        # for comp in listdir(path.join('./search_results', search_dr)) :
        #     if '_vs_' in comp : 
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
            df = pd.read_csv(inpath, sep='\t', usecols=['Run'])
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet')
            df = pd.read_parquet(inpath)
        outfile = inpath + '.triqler_output.tsv'
            # outdf = pd.read_csv(outfile, sep='\t')
        temp=pd.read_csv(outfile, sep='^', header=None,)
        l = len(temp[0][0].split('\t'))
        # print(l)
        temp2=pd.DataFrame(list(temp[0].apply(lambda x: x.split('\t', maxsplit=l-1, ))),)
        temp2.columns = temp2.loc[0, :]
        outdf = temp2.loc[1:, :].copy()
        fc = [float(x.split('diff_exp_prob_')[-1]) for x in outdf.columns if x.startswith('diff_exp_prob_')][0]
        excl = ['protein', 'peptides', ]
        for col in outdf.columns :
            if not col in excl : 
                outdf[col] = outdf[col].astype(float)
        outdf['peptides'] = outdf['peptides'].str.replace('\t', ';')
        
        outdf['FDR_pass'] = outdf['q_value'] < 0.05
        outdf['FC_pass'] = abs(outdf['log2_fold_change']) >= fc
        outdf['Organism'] = outdf['protein'].map(organism_dict)
        outdf = outdf[~outdf['protein'].str.startswith('DECOY')].copy()
        outdf = outdf.rename(columns={ 'log2_fold_change':'log2_FC', 'protein':'Protein.Group'})
        outdf['Human'] = outdf['Organism'] == 'Human'
        outdf['Ecoli'] = outdf['Organism'] == 'Ecoli'
        outdf['Yeast'] = outdf['Organism'] == 'Yeast'
        outdf['Search Engine'] = [search_engine for _ in range(len(outdf))]
        outdf['Quantitation'] = ['Triqler' for _ in range(len(outdf))]
        outdf['Imputation'] = ['Triqler' for _ in range(len(outdf))]
        outdf['FC_thr'] = [fc for _ in range(len(outdf))]
        savepath = './quantitation/'+search_dr+'_triqler.tsv'
        # print(savepath)
        makedirs(path.dirname(savepath), exist_ok=True)
        outdf.to_csv(savepath, index=False, sep='\t')

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

fragpipe24_LFQbenchTOF_diatracer_triqler 

diann181_ups_ecoli_pg_intensities 

tmp 

fragpipe24_LFQbench_umpire 

.ipynb_checkpoints 

fragpipe24_ups_ecoli_msfragger_triqler 

diann231_ups_ecoli_triqler 

fragpipe24_LFQbench_umpire_triqler 

diann231_25fmol_vs_5fmol_rerun 

fragpipe24_LFQbench.sh 

diann231_LFQbench 

fragpipe24_ups_ecoli_umpire_triqler 

fragpipe24_ups_ecoli_msfragger 

diann231_ups_ecoli 

fragpipe24_UPS-Ecoli.sh 



## LFQbench Orbitrap triqler opt fc threshold

In [161]:
min_samples = 9
python_path = ''
colstarter = 'LFQ_Orbitrap_AIF_Condition_'
colsplitter = '_Sample'
condstarter = 'Condition_'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'LFQbench' in search_dr and path.isdir(path.join('./search_results', search_dr)) and search_dr.startswith('diann') :
    if 'LFQbench' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or search_dr.startswith('diann')) :
        # for comp in listdir(path.join('./search_results', search_dr)) :
        #     if '_vs_' in comp : 
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
            df = pd.read_csv(inpath, sep='\t', usecols=['Run'])
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet')
            df = pd.read_parquet(inpath)
        print(set(df['Run'].values))
        all_columns = sorted(list(set(df['Run'].values)), 
                             key=lambda x: x.split(colstarter)[-1].split(colsplitter)[0].replace('_', '.') + x.split(colsplitter)[-1][0],
                             reverse=False)

        sample_path = inpath + '.sample_file.tsv'
        # conc_1, conc_2 = comp.replace('fmol', '').split('_vs_')
        # print(conc_1, conc_2)
        # fc = np.abs(0.5*min(np.log2(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), np.log2(10)))
        fc = 0.5
        samples = sorted([path.basename(col) for col in all_columns if col.startswith(colstarter)], 
                         reverse=False)
        temp_conds = sorted([condstarter+col.split(colstarter)[-1].split(colsplitter)[0] for col in all_columns if col.startswith(colstarter)],
                            reverse=False)
        a = pd.DataFrame([samples, temp_conds], index=['run', 'condition']).T
        # print(a)
        a.to_csv(sample_path, sep='\t', index=False)
        out_path = inpath + '.triqler_input.tsv'
        print(sample_path, out_path, inpath, sep='\n')
        !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/python -m triqler.convert.diann --file_list_file $sample_path --out_file $out_path $inpath
        infile = out_path
        outfile = infile.replace('triqler_input', 'triqler_output_opt')
        pref = 'DECOY_'
        print(outfile)
        if not 'diann' in search_dr :
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
            fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
        else :
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
            fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
        

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

{'LFQ_Orbitrap_AIF_Condition_B_Sample_Gamma_02', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Beta_02', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Beta_03', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Gamma_01', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_02', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_03', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_03', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_01', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_03', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_01', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Gamma_03', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_02', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Beta_01', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_01', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_01', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_03', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_02', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_02'}
./search_results/fragpipe24_LFQbench_msfragger_triqler/dia-quant-output/report

In [162]:
organism_dict = double_lfqbench_organism_dict
colstarter = 'LFQ_Orbitrap_AIF_Condition_'
colsplitter = '_Sample'
condstarter = 'Condition_'
fc = 0.5
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'LFQbench' in search_dr and path.isdir(path.join('./search_results', search_dr)) and search_dr.startswith('diann') :
    if 'LFQbench' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or 'diann' in search_dr) :
        # for comp in listdir(path.join('./search_results', search_dr)) :
        #     if '_vs_' in comp : 
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
            df = pd.read_csv(inpath, sep='\t', usecols=['Run'])
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet')
            df = pd.read_parquet(inpath)
        outfile = inpath + '.triqler_output_opt.tsv'
            # outdf = pd.read_csv(outfile, sep='\t')
        temp=pd.read_csv(outfile, sep='^', header=None,)
        l = len(temp[0][0].split('\t'))
        # print(l)
        temp2=pd.DataFrame(list(temp[0].apply(lambda x: x.split('\t', maxsplit=l-1, ))),)
        temp2.columns = temp2.loc[0, :]
        outdf = temp2.loc[1:, :].copy()
        fc = [float(x.split('diff_exp_prob_')[-1]) for x in outdf.columns if x.startswith('diff_exp_prob_')][0]
        excl = ['protein', 'peptides', ]
        for col in outdf.columns :
            if not col in excl : 
                outdf[col] = outdf[col].astype(float)
        outdf['peptides'] = outdf['peptides'].str.replace('\t', ';')
        
        outdf['FDR_pass'] = outdf['q_value'] < 0.05
        outdf['FC_pass'] = abs(outdf['log2_fold_change']) >= fc
        outdf['Organism'] = outdf['protein'].map(organism_dict)
        outdf = outdf[~outdf['protein'].str.startswith('DECOY')].copy()
        outdf = outdf.rename(columns={ 'log2_fold_change':'log2_FC', 'protein':'Protein.Group'})
        outdf['Human'] = outdf['Organism'] == 'Human'
        outdf['Ecoli'] = outdf['Organism'] == 'Ecoli'
        outdf['Yeast'] = outdf['Organism'] == 'Yeast'
        outdf['Search Engine'] = [search_engine for _ in range(len(outdf))]
        outdf['Quantitation'] = ['Triqler' for _ in range(len(outdf))]
        outdf['Imputation'] = ['Triqler' for _ in range(len(outdf))]
        outdf['FC_thr'] = [fc for _ in range(len(outdf))]
        savepath = './quantitation/'+search_dr+'_triqler_opt.tsv'
        # print(savepath)
        makedirs(path.dirname(savepath), exist_ok=True)
        outdf.to_csv(savepath, index=False, sep='\t')

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

fragpipe24_LFQbenchTOF_diatracer_triqler 

diann181_ups_ecoli_pg_intensities 

tmp 

fragpipe24_LFQbench_umpire 

.ipynb_checkpoints 

fragpipe24_ups_ecoli_msfragger_triqler 

diann231_ups_ecoli_triqler 

fragpipe24_LFQbench_umpire_triqler 

diann231_25fmol_vs_5fmol_rerun 

fragpipe24_LFQbench.sh 

diann231_LFQbench 

fragpipe24_ups_ecoli_umpire_triqler 

fragpipe24_ups_ecoli_msfragger 

diann231_ups_ecoli 

fragpipe24_UPS-Ecoli.sh 



In [127]:
np.abs(-5) > 0.05

np.True_

## LFQbench Orbitrap triqler opt fc threshold minsamples 6

In [16]:
min_samples = 6
python_path = ''
colstarter = 'LFQ_Orbitrap_AIF_Condition_'
colsplitter = '_Sample'
condstarter = 'Condition_'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'LFQbench' in search_dr and path.isdir(path.join('./search_results', search_dr)) and search_dr.startswith('diann') :
    if 'LFQbench' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or search_dr.startswith('diann')) :
        # for comp in listdir(path.join('./search_results', search_dr)) :
        #     if '_vs_' in comp : 
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
            df = pd.read_csv(inpath, sep='\t', usecols=['Run'])
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet')
            df = pd.read_parquet(inpath)
        print(set(df['Run'].values))
        all_columns = sorted(list(set(df['Run'].values)), 
                             key=lambda x: x.split(colstarter)[-1].split(colsplitter)[0].replace('_', '.') + x.split(colsplitter)[-1][0],
                             reverse=False)

        sample_path = inpath + '.sample_file.tsv'
        # conc_1, conc_2 = comp.replace('fmol', '').split('_vs_')
        # print(conc_1, conc_2)
        # fc = np.abs(0.5*min(np.log2(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), np.log2(10)))
        fc = 0.5
        samples = sorted([path.basename(col) for col in all_columns if col.startswith(colstarter)], 
                         reverse=False)
        temp_conds = sorted([condstarter+col.split(colstarter)[-1].split(colsplitter)[0] for col in all_columns if col.startswith(colstarter)],
                            reverse=False)
        a = pd.DataFrame([samples, temp_conds], index=['run', 'condition']).T
        # print(a)
        a.to_csv(sample_path, sep='\t', index=False)
        out_path = inpath + '.triqler_input.tsv'
        print(sample_path, out_path, inpath, sep='\n')
        !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/python -m triqler.convert.diann --file_list_file $sample_path --out_file $out_path $inpath
        infile = out_path
        outfile = infile.replace('triqler_input', 'triqler_output_opt_mins{}'.format(min_samples))
        pref = 'DECOY_'
        print(outfile)
        if not 'diann' in search_dr :
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
            fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
        else :
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
            fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

{'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_02', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_03', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_01', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_02', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_03', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_01', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_01', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Beta_02', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_01', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Gamma_01', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_03', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_02', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Beta_03', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Gamma_03', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_03', 'LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_02', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Gamma_02', 'LFQ_Orbitrap_AIF_Condition_B_Sample_Beta_01'}
./search_results/fragpipe24_LFQbench_msfragger_triqler/dia-quant-output/report

In [17]:
organism_dict = double_lfqbench_organism_dict
colstarter = 'LFQ_Orbitrap_AIF_Condition_'
colsplitter = '_Sample'
condstarter = 'Condition_'
fc = 0.5
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'LFQbench' in search_dr and path.isdir(path.join('./search_results', search_dr)) and search_dr.startswith('diann') :
    if 'LFQbench' in search_dr and not 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or 'diann' in search_dr) :
        # for comp in listdir(path.join('./search_results', search_dr)) :
        #     if '_vs_' in comp : 
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
            df = pd.read_csv(inpath, sep='\t', usecols=['Run'])
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet')
            df = pd.read_parquet(inpath)
        outfile = inpath + '.triqler_output_opt_mins6.tsv'
            # outdf = pd.read_csv(outfile, sep='\t')
        temp=pd.read_csv(outfile, sep='^', header=None,)
        l = len(temp[0][0].split('\t'))
        # print(l)
        temp2=pd.DataFrame(list(temp[0].apply(lambda x: x.split('\t', maxsplit=l-1, ))),)
        temp2.columns = temp2.loc[0, :]
        outdf = temp2.loc[1:, :].copy()
        fc = [float(x.split('diff_exp_prob_')[-1]) for x in outdf.columns if x.startswith('diff_exp_prob_')][0]
        excl = ['protein', 'peptides', ]
        for col in outdf.columns :
            if not col in excl : 
                outdf[col] = outdf[col].astype(float)
        outdf['peptides'] = outdf['peptides'].str.replace('\t', ';')
        outdf['FDR_pass'] = outdf['q_value'] < 0.05
        outdf['FC_pass'] = abs(outdf['log2_fold_change']) >= fc
        outdf['Organism'] = outdf['protein'].map(organism_dict)
        outdf = outdf[~outdf['protein'].str.startswith('DECOY')].copy()
        outdf = outdf.rename(columns={ 'log2_fold_change':'log2_FC', 'protein':'Protein.Group'})
        outdf['Human'] = outdf['Organism'] == 'Human'
        outdf['Ecoli'] = outdf['Organism'] == 'Ecoli'
        outdf['Yeast'] = outdf['Organism'] == 'Yeast'
        try :
            se = search_dr.split('LFQbench')[-1].split('_')[1]
        except :
            se = search_dr.split('_')[0]
        outdf['Search Engine'] = [se for _ in range(len(outdf))]
        outdf['Quantitation'] = ['Triqler' for _ in range(len(outdf))]
        outdf['Imputation'] = ['Triqler' for _ in range(len(outdf))]
        outdf['FC_thr'] = [fc for _ in range(len(outdf))]
        savepath = './quantitation/'+search_dr+'_triqler_opt_mins6.tsv'
        # print(savepath)
        makedirs(path.dirname(savepath), exist_ok=True)
        outdf.to_csv(savepath, index=False, sep='\t')

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

fragpipe24_LFQbenchTOF_diatracer_triqler 

diann181_ups_ecoli_pg_intensities 

tmp 

fragpipe24_LFQbench_umpire 

.ipynb_checkpoints 

fragpipe24_ups_ecoli_msfragger_triqler 

diann231_ups_ecoli_triqler 

fragpipe24_LFQbench_umpire_triqler 

diann231_25fmol_vs_5fmol_rerun 

fragpipe24_LFQbench.sh 

diann231_LFQbench 

fragpipe24_ups_ecoli_umpire_triqler 

fragpipe24_ups_ecoli_msfragger 

diann231_ups_ecoli 

fragpipe24_UPS-Ecoli.sh 



# LFQbench Bruker timsTOF Pro

## Standart min_samples 9

In [ ]:
min_samples = 9
colstarter = 'LFQ_timsTOFPro_diaPASEF_Condition_'
colsplitter = '_Sample'
condstarter = 'Condition_'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'LFQbench' in search_dr and path.isdir(path.join('./search_results', search_dr)) and search_dr.startswith('diann') :
    if 'LFQbench' in search_dr and 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or search_dr.startswith('diann')) :
        # for comp in listdir(path.join('./search_results', search_dr)) :
        #     if '_vs_' in comp : 
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
            df = pd.read_csv(inpath, sep='\t', usecols=['Run'])
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet')
            df = pd.read_parquet(inpath)
        print(set(df['Run'].values))
        all_columns = sorted(list(set(df['Run'].values)), 
                             key=lambda x: x.split(colstarter)[-1].split(colsplitter)[0].replace('_', '.') + x.split(colsplitter)[-1][0],
                             reverse=False)

        sample_path = inpath + '.sample_file.tsv'
        # conc_1, conc_2 = comp.replace('fmol', '').split('_vs_')
        # print(conc_1, conc_2)
        # fc = np.abs(0.5*min(np.log2(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), np.log2(10)))
        fc = 0.5
        samples = sorted([path.basename(col) for col in all_columns if col.startswith(colstarter)], 
                         reverse=False)
        temp_conds = sorted([condstarter+col.split(colstarter)[-1].split(colsplitter)[0] for col in all_columns if col.startswith(colstarter)],
                            reverse=False)
        a = pd.DataFrame([samples, temp_conds], index=['run', 'condition']).T
        print(a)
        a.to_csv(sample_path, sep='\t', index=False)
        out_path = inpath + '.triqler_input.tsv'
        print(sample_path, out_path, inpath, sep='\n')
        !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/python -m triqler.convert.diann --file_list_file $sample_path --out_file $out_path $inpath
        infile = out_path
        outfile = infile.replace('triqler_input', 'triqler_output')
        pref = 'DECOY_'
        print(outfile)
        if not 'diann' in search_dr :
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
            # fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
            # output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
        else :
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
            # fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
            # output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
       

In [ ]:
organism_dict = double_lfqbench_organism_dict
colstarter = 'LFQ_timsTOFPro_diaPASEF_Condition_'
# colstarter = 'LFQ_Orbitrap_AIF_Condition_'
colsplitter = '_Sample'
condstarter = 'Condition_'
fc = 0.5
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    try :
        search_engine = search_dr.split('LFQbench')[-1].split('_')[1]
    except :
        search_engine = search_dr.split('_')[0]
    # if 'LFQbench' in search_dr and path.isdir(path.join('./search_results', search_dr)) and search_dr.startswith('diann') :
    if 'LFQbench' in search_dr and 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or 'diann' in search_dr) :
        # for comp in listdir(path.join('./search_results', search_dr)) :
        #     if '_vs_' in comp : 
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
            df = pd.read_csv(inpath, sep='\t', usecols=['Run'])
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet')
            df = pd.read_parquet(inpath)
        outfile = inpath + '.triqler_output_st_mins9.tsv'
            # outdf = pd.read_csv(outfile, sep='\t')
        temp=pd.read_csv(outfile, sep='^', header=None,)
        l = len(temp[0][0].split('\t'))
        # print(l)
        temp2=pd.DataFrame(list(temp[0].apply(lambda x: x.split('\t', maxsplit=l-1, ))),)
        temp2.columns = temp2.loc[0, :]
        outdf = temp2.loc[1:, :].copy()
        fc = [float(x.split('diff_exp_prob_')[-1]) for x in outdf.columns if x.startswith('diff_exp_prob_')][0]
        excl = ['protein', 'peptides', ]
        for col in outdf.columns :
            if not col in excl : 
                outdf[col] = outdf[col].astype(float)
        outdf['peptides'] = outdf['peptides'].str.replace('\t', ';')
        
        outdf['FDR_pass'] = outdf['q_value'] < 0.05
        outdf['FC_pass'] = abs(outdf['log2_fold_change']) >= fc
        outdf['Organism'] = outdf['protein'].map(organism_dict)
        outdf = outdf[~outdf['protein'].str.startswith('DECOY')].copy()
        outdf = outdf.rename(columns={ 'log2_fold_change':'log2_FC', 'protein':'Protein.Group'})
        outdf['Human'] = outdf['Organism'] == 'Human'
        outdf['Ecoli'] = outdf['Organism'] == 'Ecoli'
        outdf['Yeast'] = outdf['Organism'] == 'Yeast'
        outdf['Search Engine'] = [search_engine for _ in range(len(outdf))]
        outdf['Quantitation'] = ['Triqler' for _ in range(len(outdf))]
        outdf['Imputation'] = ['Triqler' for _ in range(len(outdf))]
        outdf['FC_thr'] = [fc for _ in range(len(outdf))]
        savepath = './quantitation/'+search_dr+'_triqler_output.tsv'
        # print(savepath)
        makedirs(path.dirname(savepath), exist_ok=True)
        outdf.to_csv(savepath, index=False, sep='\t')

## opt min_samples 3

In [163]:
min_samples = 3
colstarter = 'LFQ_timsTOFPro_diaPASEF_Condition_'
colsplitter = '_Sample'
condstarter = 'Condition_'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'LFQbench' in search_dr and path.isdir(path.join('./search_results', search_dr)) and search_dr.startswith('diann') :
    if 'LFQbench' in search_dr and 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or search_dr.startswith('diann')) :
        # for comp in listdir(path.join('./search_results', search_dr)) :
        #     if '_vs_' in comp : 
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
            df = pd.read_csv(inpath, sep='\t', usecols=['Run'])
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet')
            df = pd.read_parquet(inpath)
        print(set(df['Run'].values))
        all_columns = sorted(list(set(df['Run'].values)), 
                             key=lambda x: x.split(colstarter)[-1].split(colsplitter)[0].replace('_', '.') + x.split(colsplitter)[-1][0],
                             reverse=False)

        sample_path = inpath + '.sample_file.tsv'
        # conc_1, conc_2 = comp.replace('fmol', '').split('_vs_')
        # print(conc_1, conc_2)
        # fc = np.abs(0.5*min(np.log2(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), np.log2(10)))
        fc = 0.5
        samples = sorted([path.basename(col) for col in all_columns if col.startswith(colstarter)], 
                         reverse=False)
        temp_conds = sorted([condstarter+col.split(colstarter)[-1].split(colsplitter)[0] for col in all_columns if col.startswith(colstarter)],
                            reverse=False)
        a = pd.DataFrame([samples, temp_conds], index=['run', 'condition']).T
        print(a)
        a.to_csv(sample_path, sep='\t', index=False)
        out_path = inpath + '.triqler_input.tsv'
        print(sample_path, out_path, inpath, sep='\n')
        !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/python -m triqler.convert.diann --file_list_file $sample_path --out_file $out_path $inpath
        infile = out_path
        outfile = infile.replace('triqler_input', 'triqler_output_opt')
        pref = 'DECOY_'
        print(outfile)
        if not 'diann' in search_dr :
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
            fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
        else :
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
            fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
       

diann231_LFQbenchTOF 

{'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_01', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_02', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Gamma_03', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_03', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Gamma_02', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_02', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_03', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_01', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_02', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Gamma_01', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_01', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_01', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_01', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_02', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_03', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_02', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_03', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gam

In [164]:
organism_dict = double_lfqbench_organism_dict
colstarter = 'LFQ_timsTOFPro_diaPASEF_Condition_'
# colstarter = 'LFQ_Orbitrap_AIF_Condition_'
colsplitter = '_Sample'
condstarter = 'Condition_'
fc = 0.5
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    try :
        search_engine = search_dr.split('LFQbench')[-1].split('_')[1]
    except :
        search_engine = search_dr.split('_')[0]
    # if 'LFQbench' in search_dr and path.isdir(path.join('./search_results', search_dr)) and search_dr.startswith('diann') :
    if 'LFQbench' in search_dr and 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or 'diann' in search_dr) :
        # for comp in listdir(path.join('./search_results', search_dr)) :
        #     if '_vs_' in comp : 
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
            df = pd.read_csv(inpath, sep='\t', usecols=['Run'])
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet')
            df = pd.read_parquet(inpath)
        outfile = inpath + '.triqler_output_opt.tsv'
            # outdf = pd.read_csv(outfile, sep='\t')
        temp=pd.read_csv(outfile, sep='^', header=None,)
        l = len(temp[0][0].split('\t'))
        # print(l)
        temp2=pd.DataFrame(list(temp[0].apply(lambda x: x.split('\t', maxsplit=l-1, ))),)
        temp2.columns = temp2.loc[0, :]
        outdf = temp2.loc[1:, :].copy()
        fc = [float(x.split('diff_exp_prob_')[-1]) for x in outdf.columns if x.startswith('diff_exp_prob_')][0]
        excl = ['protein', 'peptides', ]
        for col in outdf.columns :
            if not col in excl : 
                outdf[col] = outdf[col].astype(float)
        outdf['peptides'] = outdf['peptides'].str.replace('\t', ';')
        
        outdf['FDR_pass'] = outdf['q_value'] < 0.05
        outdf['FC_pass'] = abs(outdf['log2_fold_change']) >= fc
        outdf['Organism'] = outdf['protein'].map(organism_dict)
        outdf = outdf[~outdf['protein'].str.startswith('DECOY')].copy()
        outdf = outdf.rename(columns={ 'log2_fold_change':'log2_FC', 'protein':'Protein.Group'})
        outdf['Human'] = outdf['Organism'] == 'Human'
        outdf['Ecoli'] = outdf['Organism'] == 'Ecoli'
        outdf['Yeast'] = outdf['Organism'] == 'Yeast'
        outdf['Search Engine'] = [search_engine for _ in range(len(outdf))]
        outdf['Quantitation'] = ['Triqler' for _ in range(len(outdf))]
        outdf['Imputation'] = ['Triqler' for _ in range(len(outdf))]
        outdf['FC_thr'] = [fc for _ in range(len(outdf))]
        savepath = './quantitation/'+search_dr+'_triqler_opt.tsv'
        # print(savepath)
        makedirs(path.dirname(savepath), exist_ok=True)
        outdf.to_csv(savepath, index=False, sep='\t')

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

fragpipe24_LFQbenchTOF_diatracer_triqler 

diann181_ups_ecoli_pg_intensities 

tmp 

fragpipe24_LFQbench_umpire 

.ipynb_checkpoints 

fragpipe24_ups_ecoli_msfragger_triqler 

diann231_ups_ecoli_triqler 

fragpipe24_LFQbench_umpire_triqler 

diann231_25fmol_vs_5fmol_rerun 

fragpipe24_LFQbench.sh 

diann231_LFQbench 

fragpipe24_ups_ecoli_umpire_triqler 

fragpipe24_ups_ecoli_msfragger 

diann231_ups_ecoli 

fragpipe24_UPS-Ecoli.sh 



## opt min_samples 6

In [ ]:
min_samples = 6
colstarter = 'LFQ_timsTOFPro_diaPASEF_Condition_'
colsplitter = '_Sample'
condstarter = 'Condition_'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'LFQbench' in search_dr and path.isdir(path.join('./search_results', search_dr)) and search_dr.startswith('diann') :
    if 'LFQbench' in search_dr and 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or search_dr.startswith('diann')) :
        # for comp in listdir(path.join('./search_results', search_dr)) :
        #     if '_vs_' in comp : 
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
            df = pd.read_csv(inpath, sep='\t', usecols=['Run'])
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet')
            df = pd.read_parquet(inpath)
        print(set(df['Run'].values))
        all_columns = sorted(list(set(df['Run'].values)), 
                             key=lambda x: x.split(colstarter)[-1].split(colsplitter)[0].replace('_', '.') + x.split(colsplitter)[-1][0],
                             reverse=False)

        sample_path = inpath + '.sample_file.tsv'
        # conc_1, conc_2 = comp.replace('fmol', '').split('_vs_')
        # print(conc_1, conc_2)
        # fc = np.abs(0.5*min(np.log2(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), np.log2(10)))
        fc = 0.5
        samples = sorted([path.basename(col) for col in all_columns if col.startswith(colstarter)], 
                         reverse=False)
        temp_conds = sorted([condstarter+col.split(colstarter)[-1].split(colsplitter)[0] for col in all_columns if col.startswith(colstarter)],
                            reverse=False)
        a = pd.DataFrame([samples, temp_conds], index=['run', 'condition']).T
        print(a)
        a.to_csv(sample_path, sep='\t', index=False)
        out_path = inpath + '.triqler_input.tsv'
        print(sample_path, out_path, inpath, sep='\n')
        !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/python -m triqler.convert.diann --file_list_file $sample_path --out_file $out_path $inpath
        infile = out_path
        outfile = infile.replace('triqler_input', 'triqler_output_opt_mins6')
        pref = 'DECOY_'
        print(outfile)
        if not 'diann' in search_dr :
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
            fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
        else :
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
            fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile

diann231_LFQbenchTOF 

{'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_01', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_03', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Gamma_02', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_02', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_03', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_01', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_01', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_02', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_01', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_03', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_03', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_03', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_02', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Gamma_01', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_02', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_01', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Gamma_03', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gam

In [ ]:
organism_dict = double_lfqbench_organism_dict
colstarter = 'LFQ_timsTOFPro_diaPASEF_Condition_'
# colstarter = 'LFQ_Orbitrap_AIF_Condition_'
colsplitter = '_Sample'
condstarter = 'Condition_'
fc = 0.5
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    try :
        search_engine = search_dr.split('LFQbench')[-1].split('_')[1]
    except :
        search_engine = search_dr.split('_')[0]
    # if 'LFQbench' in search_dr and path.isdir(path.join('./search_results', search_dr)) and search_dr.startswith('diann') :
    if 'LFQbench' in search_dr and 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or 'diann' in search_dr) :
        # for comp in listdir(path.join('./search_results', search_dr)) :
        #     if '_vs_' in comp : 
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
            df = pd.read_csv(inpath, sep='\t', usecols=['Run'])
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet')
            df = pd.read_parquet(inpath)
        outfile = inpath + '.triqler_output_opt_mins6.tsv'
            # outdf = pd.read_csv(outfile, sep='\t')
        temp=pd.read_csv(outfile, sep='^', header=None,)
        l = len(temp[0][0].split('\t'))
        # print(l)
        temp2=pd.DataFrame(list(temp[0].apply(lambda x: x.split('\t', maxsplit=l-1, ))),)
        temp2.columns = temp2.loc[0, :]
        outdf = temp2.loc[1:, :].copy()
        fc = [float(x.split('diff_exp_prob_')[-1]) for x in outdf.columns if x.startswith('diff_exp_prob_')][0]
        excl = ['protein', 'peptides', ]
        for col in outdf.columns :
            if not col in excl : 
                outdf[col] = outdf[col].astype(float)
        outdf['peptides'] = outdf['peptides'].str.replace('\t', ';')
        
        outdf['FDR_pass'] = outdf['q_value'] < 0.05
        outdf['FC_pass'] = abs(outdf['log2_fold_change']) >= fc
        outdf['Organism'] = outdf['protein'].map(organism_dict)
        outdf = outdf[~outdf['protein'].str.startswith('DECOY')].copy()
        outdf = outdf.rename(columns={ 'log2_fold_change':'log2_FC', 'protein':'Protein.Group'})
        outdf['Human'] = outdf['Organism'] == 'Human'
        outdf['Ecoli'] = outdf['Organism'] == 'Ecoli'
        outdf['Yeast'] = outdf['Organism'] == 'Yeast'
        outdf['Search Engine'] = [search_engine for _ in range(len(outdf))]
        outdf['Quantitation'] = ['Triqler' for _ in range(len(outdf))]
        outdf['Imputation'] = ['Triqler' for _ in range(len(outdf))]
        outdf['FC_thr'] = [fc for _ in range(len(outdf))]
        savepath = './quantitation/'+search_dr+'_triqler_opt_mins6.tsv'
        # print(savepath)
        makedirs(path.dirname(savepath), exist_ok=True)
        outdf.to_csv(savepath, index=False, sep='\t')

## opt min_samples 9

In [22]:
min_samples = 9
colstarter = 'LFQ_timsTOFPro_diaPASEF_Condition_'
colsplitter = '_Sample'
condstarter = 'Condition_'
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    # if 'LFQbench' in search_dr and path.isdir(path.join('./search_results', search_dr)) and search_dr.startswith('diann') :
    if 'LFQbench' in search_dr and 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or search_dr.startswith('diann')) :
        # for comp in listdir(path.join('./search_results', search_dr)) :
        #     if '_vs_' in comp : 
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
            df = pd.read_csv(inpath, sep='\t', usecols=['Run'])
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet')
            df = pd.read_parquet(inpath)
        print(set(df['Run'].values))
        all_columns = sorted(list(set(df['Run'].values)), 
                             key=lambda x: x.split(colstarter)[-1].split(colsplitter)[0].replace('_', '.') + x.split(colsplitter)[-1][0],
                             reverse=False)

        sample_path = inpath + '.sample_file.tsv'
        # conc_1, conc_2 = comp.replace('fmol', '').split('_vs_')
        # print(conc_1, conc_2)
        # fc = np.abs(0.5*min(np.log2(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), np.log2(10)))
        fc = 0.5
        samples = sorted([path.basename(col) for col in all_columns if col.startswith(colstarter)], 
                         reverse=False)
        temp_conds = sorted([condstarter+col.split(colstarter)[-1].split(colsplitter)[0] for col in all_columns if col.startswith(colstarter)],
                            reverse=False)
        a = pd.DataFrame([samples, temp_conds], index=['run', 'condition']).T
        print(a)
        a.to_csv(sample_path, sep='\t', index=False)
        out_path = inpath + '.triqler_input.tsv'
        print(sample_path, out_path, inpath, sep='\n')
        !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/python -m triqler.convert.diann --file_list_file $sample_path --out_file $out_path $inpath
        infile = out_path
        outfile = infile.replace('triqler_input', 'triqler_output_opt_mins9')
        pref = 'DECOY_'
        print(outfile)
        if not 'diann' in search_dr :
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
            fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
        else :
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile
            fc = [float(x.split(' ')[-1]) for x in output_lines if x.startswith('Minimum advisable')][0] + 0.01
            output_lines = !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples $min_samples --fold_change_eval $fc $infile

diann231_LFQbenchTOF 

{'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_01', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_03', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Gamma_02', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_02', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_03', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_01', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_01', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_02', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_01', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_03', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_03', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_03', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_02', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Gamma_01', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_02', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_01', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Gamma_03', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gam

In [23]:
organism_dict = double_lfqbench_organism_dict
colstarter = 'LFQ_timsTOFPro_diaPASEF_Condition_'
# colstarter = 'LFQ_Orbitrap_AIF_Condition_'
colsplitter = '_Sample'
condstarter = 'Condition_'
fc = 0.5
for search_dr in listdir('./search_results') :
    print(search_dr, '\n')
    try :
        search_engine = search_dr.split('LFQbench')[-1].split('_')[1]
    except :
        search_engine = search_dr.split('_')[0]
    # if 'LFQbench' in search_dr and path.isdir(path.join('./search_results', search_dr)) and search_dr.startswith('diann') :
    if 'LFQbench' in search_dr and 'TOF' in search_dr and path.isdir(path.join('./search_results', search_dr)) and ('triqler' in search_dr or 'diann' in search_dr) :
        # for comp in listdir(path.join('./search_results', search_dr)) :
        #     if '_vs_' in comp : 
        if search_dr.startswith('fragpipe') :
            inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
            df = pd.read_csv(inpath, sep='\t', usecols=['Run'])
        else :
            inpath = path.join('./search_results', search_dr, 'report.parquet')
            df = pd.read_parquet(inpath)
        outfile = inpath + '.triqler_output_opt_mins9.tsv'
            # outdf = pd.read_csv(outfile, sep='\t')
        temp=pd.read_csv(outfile, sep='^', header=None,)
        l = len(temp[0][0].split('\t'))
        # print(l)
        temp2=pd.DataFrame(list(temp[0].apply(lambda x: x.split('\t', maxsplit=l-1, ))),)
        temp2.columns = temp2.loc[0, :]
        outdf = temp2.loc[1:, :].copy()
        fc = [float(x.split('diff_exp_prob_')[-1]) for x in outdf.columns if x.startswith('diff_exp_prob_')][0]
        excl = ['protein', 'peptides', ]
        for col in outdf.columns :
            if not col in excl : 
                outdf[col] = outdf[col].astype(float)
        outdf['peptides'] = outdf['peptides'].str.replace('\t', ';')
        
        outdf['FDR_pass'] = outdf['q_value'] < 0.05
        outdf['FC_pass'] = abs(outdf['log2_fold_change']) >= fc
        outdf['Organism'] = outdf['protein'].map(organism_dict)
        outdf = outdf[~outdf['protein'].str.startswith('DECOY')].copy()
        outdf = outdf.rename(columns={ 'log2_fold_change':'log2_FC', 'protein':'Protein.Group'})
        outdf['Human'] = outdf['Organism'] == 'Human'
        outdf['Ecoli'] = outdf['Organism'] == 'Ecoli'
        outdf['Yeast'] = outdf['Organism'] == 'Yeast'
        outdf['Search Engine'] = [search_engine for _ in range(len(outdf))]
        outdf['Quantitation'] = ['Triqler' for _ in range(len(outdf))]
        outdf['Imputation'] = ['Triqler' for _ in range(len(outdf))]
        outdf['FC_thr'] = [fc for _ in range(len(outdf))]
        savepath = './quantitation/'+search_dr+'_triqler_opt_mins9.tsv'
        # print(savepath)
        makedirs(path.dirname(savepath), exist_ok=True)
        outdf.to_csv(savepath, index=False, sep='\t')

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

fragpipe24_LFQbenchTOF_diatracer_triqler 

diann181_ups_ecoli_pg_intensities 

tmp 

fragpipe24_LFQbench_umpire 

.ipynb_checkpoints 

fragpipe24_ups_ecoli_msfragger_triqler 

diann231_ups_ecoli_triqler 

fragpipe24_LFQbench_umpire_triqler 

diann231_25fmol_vs_5fmol_rerun 

fragpipe24_LFQbench.sh 

diann231_LFQbench 

fragpipe24_ups_ecoli_umpire_triqler 

fragpipe24_ups_ecoli_msfragger 

diann231_ups_ecoli 

fragpipe24_UPS-Ecoli.sh 



In [24]:
1

1

# LFQbench Bruker timsTOF Pro lower threshold

In [53]:
# colstarter=''
# colsplitter=''
fc = 0.4
for fc in [0.26, ] :
# for fc in [0.25, 0.27, 0.29, 0.32] : #, 0.35, 0.4, 0.43, 0.46] :
    colstarter = 'LFQ_timsTOFPro_diaPASEF_Condition_'
    colsplitter = '_Sample'
    condstarter = 'Condition_'
    for search_dr in listdir('./search_results') :
        print(search_dr, '\n')
        # if path.isdir(path.join('./search_results', search_dr)) and search_dr == 'diann231_LFQbenchTOF' :
        if path.isdir(path.join('./search_results', search_dr)) and search_dr == 'fragpipe24_LFQbenchTOF_diatracer_triqler' :
            # for comp in listdir(path.join('./search_results', search_dr)) :
            #     if '_vs_' in comp : 
            if search_dr.startswith('fragpipe') :
                inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
                df = pd.read_csv(inpath, sep='\t', usecols=['Run'])
            else :
                inpath = path.join('./search_results', search_dr, 'report.parquet')
                df = pd.read_parquet(inpath)
            print(set(df['Run'].values))
            all_columns = sorted(list(set(df['Run'].values)), 
                                 key=lambda x: x.split(colstarter)[-1].split(colsplitter)[0].replace('_', '.') + x.split(colsplitter)[-1][0],
                                 reverse=False)
    
            sample_path = inpath + '.sample_file.tsv'
            # conc_1, conc_2 = comp.replace('fmol', '').split('_vs_')
            # print(conc_1, conc_2)
            # fc = np.abs(0.5*min(np.log2(float(conc_1.replace('_', '.'))/float(conc_2.replace('_', '.'))), np.log2(10)))
            # fc = 0.35
            samples = sorted([path.basename(col) for col in all_columns if col.startswith(colstarter)], 
                             reverse=False)
            temp_conds = sorted([condstarter+col.split(colstarter)[-1].split(colsplitter)[0] for col in all_columns if col.startswith(colstarter)],
                                reverse=False)
            a = pd.DataFrame([samples, temp_conds], index=['run', 'condition']).T
            print(a)
            a.to_csv(sample_path, sep='\t', index=False)
            out_path = inpath + '.triqler_input.tsv'
            print(sample_path, out_path, inpath, sep='\n')
            !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/python -m triqler.convert.diann --file_list_file $sample_path --out_file $out_path $inpath
            infile = out_path
            outfile = infile.replace('triqler_input', 'triqler_output_{}'.format(str(fc)))
            pref = 'DECOY_'
            print(outfile)
            if not 'diann' in search_dr :
                !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --decoy_pattern $pref --write_spectrum_quants --min_samples 9 --fold_change_eval $fc $infile
            else :
                !~/.pyenv/versions/3.10.11/envs/triqler31011/bin/triqler --out_file $outfile --write_spectrum_quants --min_samples 9 --fold_change_eval $fc $infile


diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

fragpipe24_LFQbenchTOF_diatracer_triqler 

{'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_01', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_03', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_02', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Gamma_02', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_01', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_01', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_02', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_02', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_03', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_01', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_03', 'LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_02', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Gamma_01', 'LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_03', 'LFQ_timsTOFPro_diaPASEF_Conditi

In [54]:
organism_dict = double_lfqbench_organism_dict
colstarter = 'LFQ_timsTOFPro_diaPASEF_Condition_'
# colstarter = 'LFQ_Orbitrap_AIF_Condition_'
colsplitter = '_Sample'
condstarter = 'Condition_'
# for fc in [0.25, 0.27, 0.29, 0.32] :
for fc in [0.26,] :
    # fc = 0.35
    for search_dr in listdir('./search_results') :
        print(search_dr, '\n')
        try :
            search_engine = search_dr.split('LFQbench')[-1].split('_')[1]
        except :
            search_engine = search_dr.split('_')[0]
        if path.isdir(path.join('./search_results', search_dr)) and search_dr == 'fragpipe24_LFQbenchTOF_diatracer_triqler' :
        # if path.isdir(path.join('./search_results', search_dr)) and search_dr == 'diann231_LFQbenchTOF' :
            # for comp in listdir(path.join('./search_results', search_dr)) :
            #     if '_vs_' in comp : 
            if search_dr.startswith('fragpipe') :
                inpath = path.join('./search_results', search_dr, 'dia-quant-output', 'report.tsv')
                df = pd.read_csv(inpath, sep='\t', usecols=['Run'])
            else :
                inpath = path.join('./search_results', search_dr, 'report.parquet')
                df = pd.read_parquet(inpath)
            outfile = inpath + '.triqler_output_{}.tsv'.format(str(fc))
                # outdf = pd.read_csv(outfile, sep='\t')
            temp=pd.read_csv(outfile, sep='^', header=None,)
            l = len(temp[0][0].split('\t'))
            # print(l)
            temp2=pd.DataFrame(list(temp[0].apply(lambda x: x.split('\t', maxsplit=l-1, ))),)
            temp2.columns = temp2.loc[0, :]
            outdf = temp2.loc[1:, :].copy()
            excl = ['protein', 'peptides', ]
            for col in outdf.columns :
                if not col in excl : 
                    outdf[col] = outdf[col].astype(float)
            outdf['peptides'] = outdf['peptides'].str.replace('\t', ';')
            
            outdf['FDR_pass'] = outdf['q_value'] < 0.05
            outdf['FC_pass'] = abs(outdf['log2_fold_change']) >= fc
            outdf['Organism'] = outdf['protein'].map(organism_dict)
            outdf = outdf[~outdf['protein'].str.startswith('DECOY')].copy()
            outdf = outdf.rename(columns={ 'log2_fold_change':'log2_FC', 'protein':'Protein.Group'})
            outdf['Human'] = outdf['Organism'] == 'Human'
            outdf['Ecoli'] = outdf['Organism'] == 'Ecoli'
            outdf['Yeast'] = outdf['Organism'] == 'Yeast'
            outdf['Search Engine'] = [search_engine for _ in range(len(outdf))]
            outdf['Quantitation'] = ['Triqler' for _ in range(len(outdf))]
            outdf['Imputation'] = ['Triqler' for _ in range(len(outdf))]
            outdf['FC_thr'] = [fc for _ in range(len(outdf))]
            savepath = './quantitation/'+search_dr+'_triqler_{}.tsv'.format(str(fc).replace('.', ''))
            # print(savepath)
            makedirs(path.dirname(savepath), exist_ok=True)
            outdf.to_csv(savepath, index=False, sep='\t')

diann231_LFQbenchTOF 

fragpipe24_LFQbench_msfragger_triqler 

fragpipe24_ups_ecoli_umpire 

fragpipe24_LFQbenchTOF_diatracer 

fragpipe24_LFQbench_msfragger 

fragpipe24_LFQbenchTOF_diatracer_triqler 

diann181_ups_ecoli_pg_intensities 

tmp 

fragpipe24_LFQbench_umpire 

.ipynb_checkpoints 

fragpipe24_ups_ecoli_msfragger_triqler 

diann231_ups_ecoli_triqler 

fragpipe24_LFQbench_umpire_triqler 

diann231_25fmol_vs_5fmol_rerun 

fragpipe24_LFQbench.sh 

diann231_LFQbench 

fragpipe24_ups_ecoli_umpire_triqler 

fragpipe24_ups_ecoli_msfragger 

diann231_ups_ecoli 

fragpipe24_UPS-Ecoli.sh 



In [55]:
1

1